In [3]:
import sys
import os
import pandas as pd
import numpy as np

# Add src to path
# sys.path.append(os.path.join(os.path.dirname(__file__), 'src'))

from src.data_loader import load_data
from src.solver import solve_lam_stqp

In [4]:
# Parameters
DATA_FILE = 'data/topix-large500.csv'
TARGET_RETURN = 0.0005  # 0.05% daily return (approx 12% annual)
CARDINALITY_K = 10      # Max 10 assets
MIN_WEIGHT = 0.01       # Min 1% per asset
MAX_WEIGHT = 1.0        # Max 100% per asset
PENALTY_M = 1000.0      # Penalty for return deviation

print("=== Portfolio Optimization (LAM/StQP) ===")
print(f"Target Return: {TARGET_RETURN:.6f}")
print(f"Cardinality K: {CARDINALITY_K}")
print(f"Penalty M: {PENALTY_M}")

=== Portfolio Optimization (LAM/StQP) ===
Target Return: 0.000500
Cardinality K: 10
Penalty M: 1000.0


In [5]:
# 1. Load Data
mu, sigma, assets = load_data(DATA_FILE)
print(mu)
print()
print(sigma)
print()
print(assets)

Loading data from data/topix-large500.csv...
Data loaded successfully. 495 assets, 997 time periods.
2914    0.001011
3382    0.000427
4063    0.000396
4502    0.000379
4568    0.000538
          ...   
9861    0.000424
9934    0.000642
9962   -0.000447
9987    0.000741
9989    0.000297
Length: 495, dtype: float64

          2914      3382      4063      4502      4568      6098      6367  \
2914  0.000174  0.000075  0.000098  0.000057  0.000099  0.000086  0.000089   
3382  0.000075  0.000365  0.000096  0.000062  0.000091  0.000094  0.000093   
4063  0.000098  0.000096  0.000413  0.000076  0.000175  0.000281  0.000201   
4502  0.000057  0.000062  0.000076  0.000136  0.000127  0.000083  0.000066   
4568  0.000099  0.000091  0.000175  0.000127  0.000593  0.000188  0.000154   
...        ...       ...       ...       ...       ...       ...       ...   
9861  0.000038  0.000055  0.000050  0.000030  0.000040  0.000055  0.000043   
9934  0.000080  0.000068  0.000119  0.000061  0.000097  0.0

In [6]:
# 2. Solve
result = solve_lam_stqp(
    mu, sigma, 
    target_return=TARGET_RETURN,
    k_max=CARDINALITY_K,
    min_weight=MIN_WEIGHT,
    max_weight=MAX_WEIGHT,
    penalty_m=PENALTY_M
)

# 3. Output Results
print("\n=== Optimization Results ===")
print(f"Objective Value: {result['objective']:.6f}")
print(f"Portfolio Return: {result['return']:.6f}")
print(f"Portfolio Risk (Std): {result['risk']:.6f}")
print(f"Number of Assets: {len(result['assets'])}")
print("\nSelected Assets and Weights:")
print(result['weights'].sort_values(ascending=False))

# Save to CSV
result['weights'].to_csv('optimization_result.csv', header=['Weight'])
print("\nResults saved to optimization_result.csv")

Starting optimization with K=10, Target=0.0005, M=1000.0
  Step k=1/10...
    New best found at k=1: Obj=0.000101
  Step k=2/10...
    New best found at k=2: Obj=0.000070
  Step k=3/10...
    New best found at k=3: Obj=0.000065
  Step k=4/10...
    k=4: Obj=0.000065 (Not better)
  Step k=5/10...
    New best found at k=5: Obj=0.000038
  Step k=6/10...
    New best found at k=6: Obj=0.000035
  Step k=7/10...
    New best found at k=7: Obj=0.000033
  Step k=8/10...
    New best found at k=8: Obj=0.000031
  Step k=9/10...
    New best found at k=9: Obj=0.000030
  Step k=10/10...
    New best found at k=10: Obj=0.000029

=== Optimization Results ===
Objective Value: 0.000029
Portfolio Return: 0.000485
Portfolio Risk (Std): 0.005410
Number of Assets: 10

Selected Assets and Weights:
8729    0.200800
9023    0.192871
9434    0.145207
4665    0.144761
5016    0.108649
8267    0.077792
9072    0.073287
5838    0.023404
6417    0.022869
5803    0.010360
dtype: float64

Results saved to optimiza